In [9]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
import pandas as pd
from tqdm.notebook import tqdm

In [10]:
ROOT_DATA_DIR = Path("/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041")
CALIBRATION_RESULTS_DIR = ROOT_DATA_DIR / "pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156features/callibration"

In [11]:
# Load test predictions for all deconvolvers and calibration methods
DECONVOLVER_NAMES = ["psls"]#["nnls", "psls", "xgb", "swn", "mlp"]
CALIBRATION_METHODS = [
    "uncalibrated",
    "linear_clip01_normalize",
    "linear_clip0_normalize",
    "linear_simplex_projection",
    "vector_scaling",
]

predictions = {}  # {deconv_name: {method: test_pred array}}
target_proportions = None

for deconv_name in DECONVOLVER_NAMES:
    deconv_dir = CALIBRATION_RESULTS_DIR / f"{deconv_name}_calibrators_and_predictions"
    if not deconv_dir.exists():
        print(f"WARNING: {deconv_dir} not found, skipping")
        continue

    predictions[deconv_name] = {}

    # Uncalibrated predictions (also contains targets)
    uncalib_path = deconv_dir / "uncalibrated_predictions.npz"
    if uncalib_path.exists():
        data = np.load(uncalib_path)
        predictions[deconv_name]["uncalibrated"] = data["test_pred"]
        # Load target proportions (same across all deconvolvers)
        if target_proportions is None:
            target_proportions = data["test_target"]

    # Linear calibrated predictions
    for norm_method in ["clip01_normalize", "clip0_normalize", "simplex_projection"]:
        pred_path = deconv_dir / f"linear_{norm_method}_predictions.npz"
        if pred_path.exists():
            data = np.load(pred_path)
            predictions[deconv_name][f"linear_{norm_method}"] = data["test_pred"]

    # Vector scaling predictions
    vs_path = deconv_dir / "vector_scaling_predictions.npz"
    if vs_path.exists():
        data = np.load(vs_path)
        predictions[deconv_name]["vector_scaling"] = data["test_pred"]

print(f"Loaded predictions for deconvolvers: {list(predictions.keys())}")
for name, methods in predictions.items():
    print(f"  {name}: {list(methods.keys())}")
print(f"Target proportions shape: {target_proportions.shape}")

Loaded predictions for deconvolvers: ['psls']
  psls: ['uncalibrated', 'linear_clip01_normalize', 'linear_clip0_normalize', 'linear_simplex_projection', 'vector_scaling']
Target proportions shape: (100000, 39)


In [12]:
from scipy.stats import bootstrap

# Bootstrap 95% BCa CI for MSE across calibration methods — PSLS only
psls_preds = predictions["psls"]

results = {}
for method, test_pred in tqdm(psls_preds.items()):
    pointwise_mse = np.mean((test_pred - target_proportions) ** 2, axis=1)
    mse_point = np.mean(pointwise_mse)

    res = bootstrap(
        (pointwise_mse,),
        statistic=np.mean,
        n_resamples=5000,
        confidence_level=0.95,
        method="percentile",
        random_state=42,
        # batch=20000
    )
    results[method] = {
        "MSE": mse_point,
        "CI_lower": res.confidence_interval.low,
        "CI_upper": res.confidence_interval.high,
    }

results_df = pd.DataFrame(results).T
results_df.index.name = "calibration_method"
results_df

  0%|          | 0/5 [00:00<?, ?it/s]

,MSE,CI_lower,CI_upper
calibration_method,,,
uncalibrated,0.000177,0.000174,0.000179
linear_clip01_normalize,0.000151,0.000150,0.000153
linear_clip0_normalize,0.000150,0.000148,0.000152
linear_simplex_projection,0.000124,0.000122,0.000125
vector_scaling,0.000194,0.000192,0.000196


In [18]:
from scipy.special import ndtri
from scipy.stats import norm

def bca_ci_mean(data, n_resamples=10_000, confidence_level=0.95, random_state=42):
    """BCa bootstrap CI for np.mean on the full dataset.

    The jackknife is computed analytically for the mean:
        jack_i = (sum(data) - data[i]) / (n - 1)
    This avoids materializing n leave-one-out arrays (the scipy OOM culprit).
    """
    rng = np.random.default_rng(random_state)
    n = len(data)
    theta_hat = np.mean(data)

    # Bootstrap distribution (full n-out-of-n resamples)
    boot_stats = np.empty(n_resamples)
    for i in range(n_resamples):
        boot_stats[i] = np.mean(data[rng.integers(0, n, size=n)])

    # Bias correction (z0)
    z0 = ndtri(np.mean(boot_stats < theta_hat))

    # Acceleration (a) — analytical jackknife for the mean
    total = np.sum(data)
    jack_stats = (total - data) / (n - 1)  # 1-D array, O(n) memory
    jack_mean = np.mean(jack_stats)
    diff = jack_mean - jack_stats
    a = np.sum(diff**3) / (6.0 * np.sum(diff**2) ** 1.5)

    # Adjusted percentiles
    alpha = (1 - confidence_level) / 2
    z_lo, z_hi = ndtri(alpha), ndtri(1 - alpha)
    q_lo = norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    q_hi = norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))

    return np.percentile(boot_stats, [q_lo * 100, q_hi * 100])


# Bootstrap 95% BCa CI for MSE — PSLS only, full dataset
psls_preds = predictions["psls"]

results_bca = {}
for method, test_pred in tqdm(psls_preds.items()):
    pointwise_mse = np.mean((test_pred - target_proportions) ** 2, axis=1)
    mse_point = np.mean(pointwise_mse)

    ci_lower, ci_upper = bca_ci_mean(pointwise_mse, n_resamples=10_000)

    results_bca[method] = {
        "MSE": mse_point,
        "CI_lower": ci_lower,
        "CI_upper": ci_upper,
    }

results_bca_df = pd.DataFrame(results_bca).T
results_bca_df.index.name = "calibration_method"
results_bca_df

  0%|          | 0/5 [00:00<?, ?it/s]

,MSE,CI_lower,CI_upper
calibration_method,,,
uncalibrated,0.000177,0.000174,0.000179
linear_clip01_normalize,0.000151,0.000149,0.000153
linear_clip0_normalize,0.000150,0.000148,0.000152
linear_simplex_projection,0.000124,0.000122,0.000125
vector_scaling,0.000194,0.000192,0.000196


In [20]:
(results_bca_df*1e4).round(2)

,MSE,CI_lower,CI_upper
calibration_method,,,
uncalibrated,1.77,1.74,1.79
linear_clip01_normalize,1.51,1.49,1.53
linear_clip0_normalize,1.50,1.48,1.52
linear_simplex_projection,1.24,1.22,1.25
vector_scaling,1.94,1.92,1.96


In [17]:
print(results_bca_df.to_markdown())

| calibration_method        |         MSE |    CI_lower |    CI_upper |
|:--------------------------|------------:|------------:|------------:|
| uncalibrated              | 0.00017669  | 0.000172865 | 0.000184377 |
| linear_clip01_normalize   | 0.00015146  | 0.000148877 | 0.00015801  |
| linear_clip0_normalize    | 0.000150124 | 0.000146031 | 0.000154553 |
| linear_simplex_projection | 0.000123722 | 0.000118919 | 0.000125277 |
| vector_scaling            | 0.0001939   | 0.000190196 | 0.000199506 |


With 10000 subsample

| calibration_method        |         MSE |    CI_lower |    CI_upper |
|:--------------------------|------------:|------------:|------------:|
| uncalibrated              | 0.00017669  | 0.000169219 | 0.000185105 |
| linear_clip01_normalize   | 0.00015146  | 0.000147027 | 0.000159654 |
| linear_clip0_normalize    | 0.000150124 | 0.000144428 | 0.000157165 |
| linear_simplex_projection | 0.000123722 | 0.000115689 | 0.000124648 |
| vector_scaling            | 0.0001939   | 0.000183966 | 0.000196544 |


With 20000 subsample

| calibration_method        |         MSE |    CI_lower |    CI_upper |
|:--------------------------|------------:|------------:|------------:|
| uncalibrated              | 0.00017669  | 0.000172865 | 0.000184377 |
| linear_clip01_normalize   | 0.00015146  | 0.000148877 | 0.00015801  |
| linear_clip0_normalize    | 0.000150124 | 0.000146031 | 0.000154553 |
| linear_simplex_projection | 0.000123722 | 0.000118919 | 0.000125277 |
| vector_scaling            | 0.0001939   | 0.000190196 | 0.000199506 |